In [42]:
import os
from dotenv import load_dotenv

#langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.agents import create_agent

In [8]:
load_dotenv()  # Load environment variables from .env file

True

In [9]:
groq_key = os.getenv("GROQ_API_KEY") # Retrieve the GROQ API key from environment variables
jina_key = os.getenv("JINA_API_KEY") # Retrieve the Jina API key from environment variables

In [10]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

In [11]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8") #utf-8 encoding is used to handle a wide range of characters and symbols, ensuring that the text is read correctly without any data loss or corruption. 

documents = loader.load() # Load the documents from the specified file path using the TextLoader. The loaded documents are stored in the 'documents' variable for further processing or analysis.

print(f"Loaded {len(documents)} documents from {DATA_FILE_PATH}") # Print the number of documents loaded and the file path for verification and debugging purposes.

print(documents)

Loaded 1 documents from data/hr_policy.txt
[Document(metadata={'source': 'data/hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months f

#### LANGCHAIN DOCUMENT:

Langchain processes everything in the form of documents.
documents variable created earlier is a object of the base class DOCUMENT and that has attributes "page content" and "metadata".

DOCUMENTS:

PAGE CONTENT

METADATA



In [12]:
len(documents)

1

In [19]:
documents[0].page_content

'COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDuring probation, employees are not eligible for paid leave, but may take unpaid 

In [20]:
documents[0].metadata

{'source': 'data/hr_policy.txt'}

In [21]:
print(f"total characters in document: {len(documents[0].page_content)}")

total characters in document: 2597


In [24]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data/hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data/hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data/hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to

In [25]:
len(chunks)

9

#### now each chunk is a document

#### chunks variable we created now holds LIST OF DOCUMENTS

In [27]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data/hr_policy.txt'}


In [33]:
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("embeddings model ready the name is", embeddings_model.model_name)

embeddings model ready the name is jina-embeddings-v2-base-en


#### STORE DATA IN VECTOR DATABASE

In [37]:
vector_store = FAISS.from_documents(chunks, embeddings_model)

print("CHUNKS ARE STORED", vector_store.index.ntotal)

CHUNKS ARE STORED 9


In [ ]:
test_query = "how many sick leaves employees get"

top_matches = vector_store.similarity_search(test_query, k=2)
print(f"query: {test_query}")
for i,match in enumerate(top_matches, start = 1):
    print(f"match {i} : {match.page_content}") #again page_content is used because vectors also operate in forms of documents

query: how many sick leaves employees get
match 1 : 1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.
match 2 : 7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


In [50]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [41]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b",\
    temperature = 0
)

llm.model

'openai/gpt-oss-120b'

#### TOOL

In [51]:
#there is no need for adding a @tool decorator because the agent will automatically convert the function into a tool.

def search_hr_policy(question: str) -> str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

In [52]:
hr_assistant = create_agent(
    model = llm,
    tools = [search_hr_policy],
    system_prompt = """ 
    
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)

In [46]:
def ask_hr_assistant(question: str) -> str:
    """send a question to the HR assistant and return the answer"""
    print("="*60)
    print(f"Question: {question}")
    print("="*60)

    response = hr_assistant.invoke({"messages" : [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print(f"Answer: {answer}")
    print("="*60)
    print()
    return answer

In [53]:
response = hr_assistant.invoke(
    {"messages" : [{"role": "user", "content": "how many sick leaves employees get"}]}
)   

In [54]:
response

{'messages': [HumanMessage(content='how many sick leaves employees get', additional_kwargs={}, response_metadata={}, id='1957c4cc-ecda-4b90-b377-769cbbfe311c'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer using search_hr_policy. Use tool.', 'tool_calls': [{'id': 'fc_cd624635-fb73-45b4-a52a-d4d32cad9785', 'function': {'arguments': '{"question":"sick leave entitlement employees get"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 207, 'total_tokens': 253, 'completion_time': 0.096176043, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.009984414, 'prompt_tokens_details': None, 'queue_time': 0.319830807, 'total_time': 0.106160457}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_02b0d31eca', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1b2fc-41e5-7211-a312-e